In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [6]:
# 1. Load the dataset
df = pd.read_csv('../data/energydata_complete.csv')

print(df.head())

                  date  Appliances  lights  ...  Tdewpoint        rv1        rv2
0  2016-01-11 17:00:00          60      30  ...        5.3  13.275433  13.275433
1  2016-01-11 17:10:00          60      30  ...        5.2  18.606195  18.606195
2  2016-01-11 17:20:00          50      30  ...        5.1  28.642668  28.642668
3  2016-01-11 17:30:00          50      40  ...        5.0  45.410389  45.410389
4  2016-01-11 17:40:00          60      40  ...        4.9  10.084097  10.084097

[5 rows x 29 columns]


In [7]:
print(df.isna().sum())

date           0
Appliances     0
lights         0
T1             0
RH_1           0
T2             0
RH_2           0
T3             0
RH_3           0
T4             0
RH_4           0
T5             0
RH_5           0
T6             0
RH_6           0
T7             0
RH_7           0
T8             0
RH_8           0
T9             0
RH_9           0
T_out          0
Press_mm_hg    0
RH_out         0
Windspeed      0
Visibility     0
Tdewpoint      0
rv1            0
rv2            0
dtype: int64


In [8]:


# 2. Extract Time Features from 'date'
df['date'] = pd.to_datetime(df['date'])
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
# Month might be less relevant if the dataset spans a short time, 
# but we'll include it for completeness.
df['month'] = df['date'].dt.month

# 3. Cyclical Encoding for Time Features
def encode_cyclical(data, col, max_val):
    data[col + '_sin'] = np.sin(2 * np.pi * data[col] / max_val)
    data[col + '_cos'] = np.cos(2 * np.pi * data[col] / max_val)
    return data.drop(columns=[col])

df = encode_cyclical(df, 'hour', 24)
df = encode_cyclical(df, 'day_of_week', 7)
df = encode_cyclical(df, 'month', 12)

# 4. Drop non-predictive columns
# 'date' is now redundant. 'rv1' and 'rv2' are random noise.
df = df.drop(columns=['date', 'rv1', 'rv2'])

# 5. Define Features and Target
X = df.drop(columns=['Appliances'])
y = df['Appliances']

# 6. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. Min-Max Scaling
# This is crucial for KNN distance calculations. 
# It scales every feature (including the sin/cos ones) to the [0, 1] range.
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), 
    columns=X_train.columns
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), 
    columns=X_test.columns
)

# 8. Reconstruct and Save
train_final = X_train_scaled.copy()
train_final['y'] = y_train.values

test_final = X_test_scaled.copy()
test_final['y'] = y_test.values

train_final.to_csv('../data/energy_preprocessed_train.csv', index=False)
test_final.to_csv('../data/energy_preprocessed_test.csv', index=False)

print("Preprocessing complete!")
print(f"Features used: {len(X.columns)}")
print(f"Target: Appliances (Wh)")

Preprocessing complete!
Features used: 31
Target: Appliances (Wh)
